# Intelligent Information Retrieval Search Server
This notebook contains the Flask web application implementing an Information Retrieval system. It features:
* **BM25** (Probabilistic Model) scoring
* **TF-IDF** (Vector Space Model) scoring
* **Rocchio Relevance Feedback** for query expansion

In [18]:
%pip install flask


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
from flask import Flask, request, render_template_string, session
import json
import math
import re
import os
from nltk.stem import PorterStemmer

app = Flask(__name__)
app.secret_key = 'ir_project_secret'  # Required for session tracking

## Load Index Data & Helper Functions

In [20]:
# Load the persistent index
with open("index.json", "r") as f:
    DATA = json.load(f)

STEMMER = PorterStemmer()

def get_tokens(text):
    return re.findall(r'\w+', text.lower())

# --- Precompute each document's TF-IDF vector norm (for TRUE cosine) ---
N = DATA['num_docs']
_norm_sq = {d: 0.0 for d in DATA['doc_lengths']}
for term, postings in DATA['index'].items():
    idf = math.log10(N / len(postings))
    for d, tf in postings.items():
        w = (1 + math.log10(tf)) * idf
        _norm_sq[d] += w * w
DOC_NORMS = {d: math.sqrt(s) if s > 0 else 0.0 for d, s in _norm_sq.items()}

## Retrieval Models (BM25 & TF-IDF)

In [21]:
def get_bm25(query_tokens):
    """Calculates BM25 scores (Probabilistic Model)."""
    scores = {}
    k1, b = 1.5, 0.75
    for token in query_tokens:
        stemmed = STEMMER.stem(token)
        if stemmed in DATA['index']:
            df = len(DATA['index'][stemmed])
            idf = math.log((DATA['num_docs'] - df + 0.5) / (df + 0.5) + 1)
            for doc_id, tf in DATA['index'][stemmed].items():
                doc_len = DATA['doc_lengths'][doc_id]
                denom = tf + k1 * (1 - b + b * (doc_len / DATA['avg_dl']))
                score = idf * (tf * (k1 + 1)) / denom
                scores[doc_id] = scores.get(doc_id, 0) + score
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:10]

# --- Length Normalized TF-IDF Vector Space Model ---
def get_tfidf(query_tokens):
    """Vector Space Model with TRUE cosine similarity (scores in 0..1)."""
    query_weights = {}
    for token in set(query_tokens):
        stemmed = STEMMER.stem(token)
        if stemmed in DATA['index']:
            idf = math.log10(DATA['num_docs'] / len(DATA['index'][stemmed]))
            query_weights[stemmed] = (1 + math.log10(query_tokens.count(token))) * idf

    q_norm = math.sqrt(sum(w**2 for w in query_weights.values())) or 1.0

    scores = {}
    for stemmed, q_weight in query_weights.items():
        idf = math.log10(DATA['num_docs'] / len(DATA['index'][stemmed]))
        for doc_id, tf in DATA['index'][stemmed].items():
            doc_tfidf = (1 + math.log10(tf)) * idf
            scores[doc_id] = scores.get(doc_id, 0) + (q_weight * doc_tfidf)

    ranked = []
    for doc_id, dot in scores.items():
        if DOC_NORMS.get(doc_id, 0) > 0:
            ranked.append((doc_id, dot / (q_norm * DOC_NORMS[doc_id])))
    return sorted(ranked, key=lambda x: x[1], reverse=True)[:10]

## Rocchio Relevance Feedback

In [22]:
# --- True Empirical Rocchio Implementation ---
def apply_rocchio(original_query_tokens, relevant_doc_ids):
    alpha, beta = 1.0, 0.75
    query_vector = {}
    
    for token in original_query_tokens:
        stemmed = STEMMER.stem(token)
        query_vector[stemmed] = query_vector.get(stemmed, 0.0) + alpha

    if not relevant_doc_ids: return list(query_vector.keys())

    # Extract high-importance descriptive vectors from target documents
    for doc_id in relevant_doc_ids:
        try:
            with open(f"data/{doc_id}.txt", "r", encoding="utf-8") as f:
                content = f.read().split("\n", 1)[1] # Skip validation header line
                tokens = re.findall(r'\w+', content.lower())
                for t in tokens[:50]: # Limit evaluation scanning profile window
                    s_token = STEMMER.stem(t)
                    query_vector[s_token] = query_vector.get(s_token, 0.0) + (beta / len(relevant_doc_ids))
        except FileNotFoundError: continue

    # Sort based on calculated weights to capture the top 12 terms
    sorted_terms = sorted(query_vector.items(), key=lambda x: x[1], reverse=True)[:12]
    return [term[0] for term in sorted_terms]

## User Interface Template

In [23]:
HTML = '''
<!DOCTYPE html>
<html>
<head>
    <title>IIR Search Server</title>
    <style>body { font-family: sans-serif; margin: 30px; line-height: 1.6; }</style>
</head>
<body>
    <h2>Intelligent Information Retrieval System</h2>
    <form action="/search" method="get">
        <input name="q" value="{{q}}" style="width: 300px; padding: 5px;">
        <select name="model">
            <option value="bm25" {% if model=='bm25' %}selected{% endif %}>BM25 (Probabilistic)</option>
            <option value="tfidf" {% if model=='tfidf' %}selected{% endif %}>TF-IDF (Normalized Cosine)</option>
        </select>
        <button type="submit">Search</button>
    </form>

    {% if results %}
    <form action="/feedback" method="post">
        <input type="hidden" name="original_query" value="{{q}}">
        <h3>Ranked Results Profile:</h3>
        {% for id, score in results %}
            <div style="margin-bottom: 10px; padding: 5px; border-bottom: 1px solid #eee;">
                <input type="checkbox" name="relevant_docs" value="{{id}}">
                <strong>Title: {{ titles[id] if (titles and id in titles) else "Doc ID: " ~ id }}</strong> 
                <span style="color:#666;">(ID: {{id}} | Metric Score: {{score|round(4)}})</span>
            </div>
        {% endfor %} <br>
        <button type="submit">Execute Rocchio Vector Feedback Shift</button>
    </form>
    {% endif %}
</body>
</html>
'''

## Web Server Endpoints

In [24]:
@app.route('/')
def home():
    # Pass an empty dictionary for titles to avoid Jinja undefined errors
    return render_template_string(HTML, results=[], q="", model="bm25", titles={})

@app.route('/search')
def search():
    query = request.args.get('q', '')
    model_type = request.args.get('model', 'bm25')
    tokens = get_tokens(query)

    if model_type == 'tfidf':
        results = get_tfidf(tokens)
    else:
        results = get_bm25(tokens)

    # Hand over the 'titles' object to the frontend rendering layout engine
    return render_template_string(HTML, results=results, q=query, model=model_type, titles=DATA.get('titles', {}))

@app.route('/feedback', methods=['POST'])
def feedback():
    original_query = request.form.get('original_query', '')
    relevant_ids = request.form.getlist('relevant_docs')

    # Apply Rocchio
    expanded_tokens = apply_rocchio(get_tokens(original_query), relevant_ids)

    # Re-run search with expanded query
    results = get_bm25(expanded_tokens)
    new_query_str = " ".join(expanded_tokens[:10])  # Show snippet of new query

    # Hand over the 'titles' object here as well to protect the feedback render view
    return render_template_string(HTML, results=results, q=new_query_str, model="bm25", titles=DATA.get('titles', {}))

## Application Execution
Run the cell below to start the Flask development server.

In [25]:
if __name__ == "__main__":
    app.run(debug=True, use_reloader=False)  

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [02/Jun/2026 01:26:28] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 01:26:38] "GET /search?q=database&model=bm25 HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 01:27:19] "GET /search?q=artificial+intelligence&model=bm25 HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 01:30:18] "GET /search?q=artificial+intelligence&model=tfidf HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 01:34:53] "POST /feedback HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 01:35:09] "GET /search?q=intellig+artifici+of+journal+research+ai+thi+is+a+list&model=tfidf HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 01:35:16] "POST /feedback HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 01:38:45] "GET /search?q=operating+system&model=bm25 HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 01:38:55] "POST /feedback HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 01:39:19] "GET /search?q=operating+system&model=tfidf HTTP/1.1" 200 -
127.0.0.1 - - [02/Jun/2026 01:39:27] "POST /feedback 